In [1]:
import cdsapi
from pathlib import Path

# All yearly files will live inside this folder
OUTPUT_DIR = Path("utci_nebraska")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Files will be saved to: {OUTPUT_DIR.resolve()}")

Files will be saved to: /lustre/work/nsco/breinsvold/utci_nebraska


In [4]:
import os
from pathlib import Path

# === EDIT THESE TWO LINES ===
CDSAPIRC_PATH = Path("/lustre/work/nsco/breinsvold/utci/nebraska.cdsapirc")  # where to store the credentials file
API_KEY = "48d1d3e8-f663-49ca-b86b-e5fca16eaaee"             # token from https://cds.climate.copernicus.eu/profile
# ============================

# Create the file if it doesn't already exist
CDSAPIRC_PATH.parent.mkdir(parents=True, exist_ok=True)
if not CDSAPIRC_PATH.exists():
    CDSAPIRC_PATH.write_text(
        "url: https://cds.climate.copernicus.eu/api\n"
        f"key: {API_KEY}\n"
    )
    CDSAPIRC_PATH.chmod(0o600)  # owner read/write only
    print(f"Wrote new credentials file: {CDSAPIRC_PATH}")
else:
    print(f"Credentials file already exists: {CDSAPIRC_PATH}")

# Tell cdsapi to read from this location
os.environ["CDSAPI_RC"] = str(CDSAPIRC_PATH)
print(f"cdsapi will use: {os.environ['CDSAPI_RC']}")

Wrote new credentials file: /lustre/work/nsco/breinsvold/utci/nebraska.cdsapirc
cdsapi will use: /lustre/work/nsco/breinsvold/utci/nebraska.cdsapirc


In [5]:
# Sanity check: your ~/.cdsapirc must exist on the supercomputer (not your laptop)
cdsapirc = Path.home() / ".cdsapirc"
assert cdsapirc.exists(), (
    f"Missing {cdsapirc} — create it with your Personal Access Token from "
    "https://cds.climate.copernicus.eu/profile before continuing."
)

client = cdsapi.Client()
print("CDS client ready.")

2026-05-17 17:05:53,880 INFO [2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).


CDS client ready.


In [10]:
def download_year(year):
    target = OUTPUT_DIR / f"utci_nebraska_{year}.zip"

    # Resume support: skip years that are already on disk
    if target.exists() and target.stat().st_size > 0:
        size_mb = target.stat().st_size / 1e6
        print(f"  {year}: already downloaded ({size_mb:.1f} MB) — skipping")
        return target

    dataset = "derived-utci-historical"
    request = {
        "variable": [
            "universal_thermal_climate_index_daily_statistics",
            "universal_thermal_climate_index_monthly_statistics",
            "universal_thermal_climate_index_seasonal_statistics",
            "universal_thermal_climate_index_yearly_statistics"
        ],
        "version": "1_1",
        "product_type": "consolidated_dataset",
        "year": [str(year)],
        "month": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12"
        ],
        "day": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15",
            "16", "17", "18",
            "19", "20", "21",
            "22", "23", "24",
            "25", "26", "27",
            "28", "29", "30",
            "31"
        ],
        "area": [43.25, -104.25, 39.75, -95.25]
    }

    print(f"  {year}: queued, waiting for CDS...")
    client.retrieve(dataset, request, str(target))
    size_mb = target.stat().st_size / 1e6
    print(f"  {year}: done ({size_mb:.1f} MB)")
    return target

In [11]:
YEARS = list(range(2005, 2026))  # 2005 through 2025 inclusive (21 years)

print(f"Starting downloads: {len(YEARS)} years ({YEARS[0]}–{YEARS[-1]})")
print(f"Destination: {OUTPUT_DIR.resolve()}")
print(f"Note: each year may take minutes to hours in the CDS queue.\n")

failures = []
for i, year in enumerate(YEARS, start=1):
    print(f"[{i}/{len(YEARS)}] Year {year}:")
    try:
        download_year(year)
    except Exception as e:
        print(f"  {year}: FAILED — {e}")
        failures.append(year)

print(f"\nFinished. Successes: {len(YEARS) - len(failures)}, failures: {failures}")

Starting downloads: 21 years (2005–2025)
Destination: /lustre/work/nsco/breinsvold/utci_nebraska
Note: each year may take minutes to hours in the CDS queue.

[1/21] Year 2005:
  2005: queued, waiting for CDS...


2026-05-17 17:15:12,731 INFO [2026-04-30T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/derived-utci-historical-timeseries?tab=overview).
2026-05-17 17:15:12,732 INFO Request ID is f8eb45a6-3fe2-46d2-9aed-a199a5906e63
2026-05-17 17:15:12,890 INFO status has been updated to accepted
2026-05-17 17:15:27,597 INFO status has been updated to running
2026-05-17 17:19:34,217 INFO status has been updated to successful


3d6b450d290030fbff7510432416fd2.zip:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

  2005: done (18.4 MB)
[2/21] Year 2006:
  2006: queued, waiting for CDS...


2026-05-17 17:19:38,604 INFO [2026-04-30T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/derived-utci-historical-timeseries?tab=overview).
2026-05-17 17:19:38,605 INFO Request ID is 36616261-c22c-4915-a62a-08df5d8a5408
2026-05-17 17:19:41,311 INFO status has been updated to accepted
2026-05-17 17:19:55,276 INFO status has been updated to running
2026-05-17 17:24:01,692 INFO status has been updated to successful


f683d00d95b30db2968cbfc9a1595ff7.zip:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

... (54 outputs left out)

  2024: done (18.4 MB)
[21/21] Year 2025:
  2025: queued, waiting for CDS...


2026-05-17 18:55:33,005 INFO [2026-04-30T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/derived-utci-historical-timeseries?tab=overview).
2026-05-17 18:55:33,005 INFO Request ID is d5e14156-a8eb-4360-8b7a-428549b235e2
2026-05-17 18:55:33,162 INFO status has been updated to accepted
2026-05-17 18:55:54,886 INFO status has been updated to running
2026-05-17 19:01:54,378 INFO status has been updated to successful


a5e0210ece5ef7d6129e3bcb6e3b097f.zip:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

  2025: done (18.6 MB)

Finished. Successes: 21, failures: []


In [14]:
files = sorted(OUTPUT_DIR.glob("utci_nebraska_*.zip"))
total_bytes = sum(f.stat().st_size for f in files)

print(f"{len(files)} files, {total_bytes/1e9:.2f} GB total\n")
for f in files:
    print(f"  {f.name}: {f.stat().st_size/1e6:.1f} MB")

21 files, 0.39 GB total

  utci_nebraska_2005.zip: 18.4 MB
  utci_nebraska_2006.zip: 18.4 MB
  utci_nebraska_2007.zip: 18.4 MB
  utci_nebraska_2008.zip: 18.4 MB
  utci_nebraska_2009.zip: 18.4 MB
  utci_nebraska_2010.zip: 18.4 MB
  utci_nebraska_2011.zip: 18.4 MB
  utci_nebraska_2012.zip: 18.4 MB
  utci_nebraska_2013.zip: 18.4 MB
  utci_nebraska_2014.zip: 18.4 MB
  utci_nebraska_2015.zip: 18.4 MB
  utci_nebraska_2016.zip: 18.4 MB
  utci_nebraska_2017.zip: 18.4 MB
  utci_nebraska_2018.zip: 18.4 MB
  utci_nebraska_2019.zip: 18.4 MB
  utci_nebraska_2020.zip: 18.4 MB
  utci_nebraska_2021.zip: 19.9 MB
  utci_nebraska_2022.zip: 19.7 MB
  utci_nebraska_2023.zip: 18.4 MB
  utci_nebraska_2024.zip: 18.4 MB
  utci_nebraska_2025.zip: 18.6 MB


In [13]:
import zipfile

extracted_count = 0
for zf in sorted(OUTPUT_DIR.glob("utci_nebraska_*.zip")):
    extract_dir = zf.with_suffix("")  # e.g. utci_nebraska_2005.zip -> utci_nebraska_2005/
    extract_dir.mkdir(exist_ok=True)
    
    with zipfile.ZipFile(zf) as z:
        z.extractall(extract_dir)
        contents = z.namelist()
    
    print(f"  {zf.name} -> {extract_dir.name}/ ({len(contents)} files)")
    extracted_count += 1

print(f"\nExtracted {extracted_count} archives.")

  utci_nebraska_2005.zip -> utci_nebraska_2005/ (382 files)
  utci_nebraska_2006.zip -> utci_nebraska_2006/ (382 files)
  utci_nebraska_2007.zip -> utci_nebraska_2007/ (382 files)
  utci_nebraska_2008.zip -> utci_nebraska_2008/ (383 files)
  utci_nebraska_2009.zip -> utci_nebraska_2009/ (382 files)
  utci_nebraska_2010.zip -> utci_nebraska_2010/ (382 files)
  utci_nebraska_2011.zip -> utci_nebraska_2011/ (382 files)
  utci_nebraska_2012.zip -> utci_nebraska_2012/ (383 files)
  utci_nebraska_2013.zip -> utci_nebraska_2013/ (382 files)
  utci_nebraska_2014.zip -> utci_nebraska_2014/ (382 files)
  utci_nebraska_2015.zip -> utci_nebraska_2015/ (382 files)
  utci_nebraska_2016.zip -> utci_nebraska_2016/ (383 files)
  utci_nebraska_2017.zip -> utci_nebraska_2017/ (382 files)
  utci_nebraska_2018.zip -> utci_nebraska_2018/ (382 files)
  utci_nebraska_2019.zip -> utci_nebraska_2019/ (382 files)
  utci_nebraska_2020.zip -> utci_nebraska_2020/ (383 files)
  utci_nebraska_2021.zip -> utci_nebrask

In [15]:
sample_year = OUTPUT_DIR / "utci_nebraska_2005"
for f in sorted(sample_year.iterdir()):
    print(f"  {f.name}: {f.stat().st_size/1e6:.2f} MB")

  ECMWF_utci_daily_stats_20050101_v1.1_con.area-subset.43.25.-95.25.39.75.-104.25.nc: 0.04 MB
  ECMWF_utci_daily_stats_20050102_v1.1_con.area-subset.43.25.-95.25.39.75.-104.25.nc: 0.04 MB
  ECMWF_utci_daily_stats_20050103_v1.1_con.area-subset.43.25.-95.25.39.75.-104.25.nc: 0.04 MB
  ECMWF_utci_daily_stats_20050104_v1.1_con.area-subset.43.25.-95.25.39.75.-104.25.nc: 0.04 MB
  ECMWF_utci_daily_stats_20050105_v1.1_con.area-subset.43.25.-95.25.39.75.-104.25.nc: 0.04 MB
  ECMWF_utci_daily_stats_20050106_v1.1_con.area-subset.43.25.-95.25.39.75.-104.25.nc: 0.04 MB
  ECMWF_utci_daily_stats_20050107_v1.1_con.area-subset.43.25.-95.25.39.75.-104.25.nc: 0.04 MB
  ECMWF_utci_daily_stats_20050108_v1.1_con.area-subset.43.25.-95.25.39.75.-104.25.nc: 0.04 MB
  ECMWF_utci_daily_stats_20050109_v1.1_con.area-subset.43.25.-95.25.39.75.-104.25.nc: 0.04 MB
  ECMWF_utci_daily_stats_20050110_v1.1_con.area-subset.43.25.-95.25.39.75.-104.25.nc: 0.04 MB
  ECMWF_utci_daily_stats_20050111_v1.1_con.area-subset.43.25